# Nasa Earth Object Tracking

In [5]:
# !pip install mysql-connector-python --Package has been installed locally

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/16.4 MB 7.3 MB/s eta 0:00:02
   ---------- ----------------------------- 4.2/16.4 MB 8.4 MB/s eta 0:00:02
   --------------- ------------------------ 6.3/16.4 MB 9.0 MB/s eta 0:00:02
   ------------------- -------------------- 8.1/16.4 MB 9.1 MB/s eta 0:00:01
   ------------------------- -------------- 10.5/16.4 MB 9.2 MB/s eta 0:00:01
   ------------------------------ --------- 12.3/16.4 MB 9.2 MB/s eta 0:00:01
   ----------------------------------- ---- 14.4/16.4 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.4 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 8.8 MB/s eta 0:00:00


In [1]:
# import necesarry package

import requests
from datetime import datetime
import mysql.connector as db

API_KEY = "xoGWlgUcEuszvmGRdVD3XAiouSSgF64Ym4lnwo9O"

url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-08&api_key={API_KEY}"

response = requests.get(url)

In [2]:
data = response.json() #getting as dictionary data

In [3]:
data.keys() #listing out the keys from dictionary

dict_keys(['links', 'element_count', 'near_earth_objects'])

In [4]:
len(data['near_earth_objects']['2024-01-02']) #finding its length

22

In [5]:
details = data['near_earth_objects']

In [8]:
asteroids_data = []
close_approach_data = []

limit = 10000
start_url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-08&api_key={API_KEY}"

while len(asteroids_data) < limit:
    response = requests.get(start_url)
    data = response.json()
    details = data['near_earth_objects']
    
    for date, info in details.items():
        for asteroid in info:
            approach = asteroid['close_approach_data'][0]
            approach_datetime = datetime.strptime(approach['close_approach_date'],'%Y-%m-%d')

            #appending for asteroid table
            asteroids_data.append((
                int(asteroid['id']),
                asteroid['name'],
                asteroid['absolute_magnitude_h'],
                asteroid['estimated_diameter']['kilometers']['estimated_diameter_min'],
                asteroid['estimated_diameter']['kilometers']['estimated_diameter_max'],
                asteroid['is_potentially_hazardous_asteroid']
            ))

            #appending for close_approach table
            close_approach_data.append((
                int(asteroid['neo_reference_id']),
                approach_datetime.date(),
                float(approach['relative_velocity']['kilometers_per_hour']),
                float(approach['miss_distance']['astronomical']),
                float(approach['miss_distance']['kilometers']),
                float(approach['miss_distance']['lunar']),
                approach['orbiting_body'] 
            ))
            if len(asteroids_data) >= limit:
                break
        if len(asteroids_data) >= limit:
            break
            
    start_url = data['links'].get('next')
    if not start_url:
        break

In [10]:
len(asteroids_data)

10000

# Mysql Connection Establish

In [11]:
# specifying mysql connection

connection = db.connect(
                        host = "localhost",
                        user = "root",
                        password = "root@123",
                        database = "nasa"
)

In [12]:
# create a cursor object

cursor = connection.cursor()

In [13]:
# create a table for asteroids data storing

cursor.execute("""
create table asteroids(
id INT,
name TEXT,
absolute_magnitude_h FLOAT,
estimated_diameter_min_km FLOAT,
estimated_diameter_max_km FLOAT,
is_potentially_hazardous_asteroid BOOLEAN
);
""")

In [14]:
# create a table for closure approach data storing

cursor.execute("""
create table close_approach(
neo_reference_id INT,
close_approach_date DATE,
relative_velocity_kmph FLOAT,
astronomical_au FLOAT,
miss_distance_km FLOAT,
miss_distance_lunar FLOAT,
orbiting_body TEXT
);
""")

In [15]:
# insert asteroids data in mysql

asteroidInsert = """
INSERT INTO asteroids
VALUES
(%s, %s, %s, %s, %s, %s)
"""

cursor.executemany(asteroidInsert,asteroids_data)

connection.commit()

In [16]:
# insert close_approach data in mysql

closeInsert = """
INSERT INTO close_approach
VALUES
(%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(closeInsert,close_approach_data)

connection.commit()

In [30]:
#closing the connection after mysql insertion
connection.close()